# Exercices (Étudiant) - Client MCP avec un LLM — VERSION CORRIGÉE

Ce notebook contenait 3 bugs qui empêchaient son exécution :
1. `StdioServerParameters(command=["python", "server.py"])` — `command` doit être une chaîne, pas une liste.
2. La fonction `stub_plan` était utilisée mais jamais définie (→ `NameError` en mode stub, le mode par défaut).
3. `errlog=True` était passé à `stdio_client`, qui attend un flux texte (`sys.stderr`), pas un booléen.

Tout est corrigé ci-dessous, avec des commentaires en français à chaque étape.

In [ ]:
# On installe les bibliothèques nécessaires :
# - mcp : le SDK Model Context Protocol
# - nest_asyncio : permet d'utiliser "await" directement dans un notebook Jupyter/Colab
# - requests : utilisé si on veut appeler un vrai LLM via une API HTTP
!pip install -q mcp nest_asyncio requests

In [ ]:

import os
from pathlib import Path

# Jeton utilisé uniquement si on branche un vrai serveur MCP en HTTP (non utilisé dans cet exercice STDIO)
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")

# Interrupteur principal : False = on utilise un LLM "factice" (stub_plan, pas de clé API nécessaire)
#                          True  = on utilise un vrai LLM via GITHUB_TOKEN (voir Exercice 5)
USE_REAL_LLM = False  # mettre True si GITHUB_TOKEN est défini dans l'environnement


In [ ]:
# Imports communs utilisés dans plusieurs exercices
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# nest_asyncio permet d'utiliser "await" dans les cellules d'un notebook,
# alors que Jupyter/Colab possède déjà sa propre boucle asyncio en cours d'exécution
nest_asyncio.apply()


In [ ]:
%%writefile server.py
# ============================================================
# server.py
# ------------------------------------------------------------
# Serveur MCP minimal "DemoServer" qui expose :
#   - trois outils (tools)   : add, multiply, greet
#   - une ressource (resource) : server://info
# ============================================================

from mcp.server.fastmcp import FastMCP

# On crée le serveur MCP et on le nomme "DemoServer"
mcp = FastMCP("DemoServer")


# ------------------------------------------------------------
# OUTIL : add
# ------------------------------------------------------------
# @mcp.tool() enregistre cette fonction comme un outil que le client
# pourra découvrir (via list_tools) et appeler (via call_tool)
@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    # On retourne la somme de a et b
    return a + b


# ------------------------------------------------------------
# OUTIL (bonus / optionnel) : multiply
# ------------------------------------------------------------
# Outil supplémentaire demandé dans la section "Optionnel" de l'énoncé
@mcp.tool()
def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    # On retourne le produit de a et b
    return a * b


# ------------------------------------------------------------
# OUTIL : greet
# ------------------------------------------------------------
@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    # On retourne un message de salutation personnalisé
    return f"Hello, {name}!"


# ------------------------------------------------------------
# RESSOURCE : server://info
# ------------------------------------------------------------
# Une ressource est une donnée en lecture seule, accessible via une URI.
# Contrairement à un outil, une ressource ne "fait" rien : elle expose
# simplement de l'information (ici, la liste des outils disponibles).
@mcp.resource("server://info")
def server_info() -> str:
    """Return basic info about this demo server."""
    return "DemoServer - outils disponibles : add, multiply, greet"


# ------------------------------------------------------------
# POINT D'ENTRÉE DU PROGRAMME
# ------------------------------------------------------------
if __name__ == "__main__":
    # On démarre la boucle du serveur (transport STDIO par défaut)
    mcp.run()

## Exercice 1 (réponse fournie)

**Question : Pourquoi le transport STDIO est-il plus simple pour le développement MCP local que HTTP ?**

Le transport STDIO (entrée/sortie standard) est plus simple que HTTP pour le développement MCP local, pour plusieurs raisons :

1. **Pas de couche réseau** : STDIO fonctionne directement entre processus parent/enfant sur la même machine. Pas besoin de gérer adresses IP, ports, pare-feu ou latence réseau.
2. **Débogage plus facile** : les entrées/sorties peuvent être inspectées directement dans le terminal, sans outils réseau complexes (sniffers, proxys, etc.).
3. **Pas de déploiement de serveur** : pas besoin d'un serveur web (Gunicorn, Uvicorn...) — le serveur MCP est un simple script Python lancé en sous-processus.
4. **Configuration minimale** : il suffit d'indiquer la commande pour lancer le script serveur, contrairement à HTTP qui demande hôte, port, éventuellement certificats SSL.
5. **Contrôle direct du processus** : le client peut démarrer/arrêter le serveur directement, plus simplement qu'un service HTTP séparé.

En résumé, STDIO supprime la complexité réseau, ce qui en fait une solution légère et directe pour développer et tester des outils MCP en local.

## Exercice 2 : Connexion et initialisation

In [ ]:
import sys

async def ex2_connect():
    # CORRIGÉ : "command" doit être une chaîne (le nom de l'exécutable),
    # PAS une liste. Les arguments supplémentaires vont dans "args".
    # On utilise sys.executable pour être sûr d'utiliser le même interpréteur
    # Python que celui qui exécute ce notebook.
    params = StdioServerParameters(command=sys.executable, args=["server.py"])

    # stdio_client lance le serveur en sous-processus et ouvre les flux
    # de lecture (r) et d'écriture (w) pour communiquer avec lui.
    # errlog=sys.stderr redirige les logs d'erreur du serveur vers la console
    # (CORRIGÉ : errlog attend un flux texte, pas un booléen)
    async with stdio_client(params, errlog=sys.stderr) as (r, w):
        # On crée une session cliente à partir de ces flux
        async with ClientSession(r, w) as session:
            # Étape obligatoire : on initialise la session (poignée de main avec le serveur)
            await session.initialize()

In [ ]:
# On exécute la fonction ex2_connect définie ci-dessus
await ex2_connect()
print("Exercice 2 : OK (connecté et initialisé)")


## Exercice 3 : Découverte des ressources et outils

In [ ]:
async def ex3_list():
    # CORRIGÉ : command doit être une chaîne, pas une liste (voir Exercice 2)
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    async with stdio_client(params, errlog=sys.stderr) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            # On liste les ressources exposées par le serveur
            resources = await session.list_resources()
            print("RESOURCES:", resources)

            # On liste les outils exposés par le serveur
            tools = await session.list_tools()
            # Pour chaque outil, on affiche son nom et ses paramètres attendus
            # (inputSchema.properties décrit les arguments, façon JSON Schema)
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

In [ ]:
# On exécute la fonction ex3_list définie ci-dessus
await ex3_list()

## Exercice 4 : Conversion des outils MCP vers le format LLM

**Question : Comment se passe la conversion vers le format "outil LLM" ?**

Cette conversion se fait côté **client**, pas côté serveur, via une fonction utilitaire (`convert_to_llm_tool`). Le serveur MCP (FastMCP) expose ses outils avec un nom, une description, et un schéma d'entrée (JSON Schema).

La fonction `convert_to_llm_tool` prend un objet `tool` MCP (obtenu via `session.list_tools()`) et le transforme en dictionnaire compatible avec le format "function calling" attendu par les fournisseurs de LLM (type OpenAI) :

1. **`"type": "function"`** : indique qu'il s'agit d'une fonction appelable.
2. **`"function": {...}`** : contient les détails :
   - **`"name"`** : le nom de l'outil MCP, réutilisé tel quel pour que le LLM sache quelle fonction appeler.
   - **`"description"`** : la description de l'outil MCP, pour aider le LLM à décider quand l'utiliser.
   - **`"parameters"`** : reprend le schéma d'entrée MCP (`properties` et `required`) pour indiquer au LLM quels arguments fournir.

En résumé, `convert_to_llm_tool` sert de "pont" entre les définitions standardisées d'un serveur MCP et le format compris par un LLM pour faire des appels de fonction.

In [ ]:

def convert_to_llm_tool(tool):
    """Transforme un outil MCP en spécification de fonction pour un LLM."""
    return {
        "type": "function",
        "function": {
            # Nom de la fonction, identique au nom de l'outil MCP
            "name": tool.name,
            # Description utilisée par le LLM pour savoir quand utiliser cet outil
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                # Les propriétés (arguments attendus) viennent du schéma MCP
                "properties": tool.inputSchema.get("properties", {}),
                # La liste des arguments obligatoires
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercice 5 : Planifier et exécuter

**Planifier et exécuter :** on utilise un LLM factice (stub) ou réel pour proposer des `tool_calls`, puis on les exécute et on affiche les résultats pour un prompt comme « Add 2 to 20 ».

In [ ]:

import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    CORRIGÉ (fonction manquante dans le notebook d'origine) :
    Planificateur "factice" (stub) qui simule un LLM.
    Au lieu d'appeler un vrai modèle, on analyse le texte du prompt
    avec des mots-clés simples pour deviner quel outil appeler,
    et avec quels arguments.
    """
    text = prompt.lower()

    # On extrait tous les nombres entiers présents dans le prompt
    numbers = [int(n) for n in re.findall(r"-?\d+", prompt)]

    # On construit un dictionnaire {nom_outil: spec_outil} pour un accès facile
    available = {f["function"]["name"]: f["function"] for f in functions}

    # Cas 1 : addition (mots-clés "add", "plus", "sum")
    if any(k in text for k in ["add", "plus", "sum"]) and "add" in available and len(numbers) >= 2:
        return [{"name": "add", "args": {"a": numbers[0], "b": numbers[1]}}]

    # Cas 2 : multiplication (mots-clés "multiply", "times", "product")
    if any(k in text for k in ["multiply", "times", "product"]) and "multiply" in available and len(numbers) >= 2:
        return [{"name": "multiply", "args": {"a": numbers[0], "b": numbers[1]}}]

    # Cas 3 : salutation (mots-clés "greet", "hello", "bonjour")
    if any(k in text for k in ["greet", "hello", "bonjour"]) and "greet" in available:
        # On essaie d'extraire un nom juste après le mot-clé
        match = re.search(r"(?:greet|hello|bonjour)\s+([A-Za-z]+)", prompt, re.IGNORECASE)
        name = match.group(1) if match else "there"
        return [{"name": "greet", "args": {"name": name}}]

    # Aucun mot-clé reconnu : on ne propose aucun appel d'outil
    return []


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Décide quels outils appeler pour répondre au prompt."""
    # Mode par défaut : on utilise le planificateur factice (aucune clé API requise)
    if not use_real:
        return stub_plan(prompt, functions)

    # Mode "vrai LLM" : nécessite un jeton GITHUB_TOKEN dans l'environnement
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")

    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    # On crée un client vers le service d'inférence (modèle GPT-4o via GitHub Models)
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))

    # On envoie le prompt et la liste des outils disponibles au LLM
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."}, {"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )

    # On extrait les appels d'outils proposés par le LLM
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        # Les arguments peuvent arriver en texte JSON : on les parse si nécessaire
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    # CORRIGÉ : command doit être une chaîne, pas une liste (voir Exercice 2)
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    async with stdio_client(params, errlog=sys.stderr) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            # On récupère la liste des outils exposés par le serveur
            tools = await session.list_tools()

            # On convertit chaque outil MCP au format attendu par le LLM
            functions = [convert_to_llm_tool(t) for t in tools.tools]

            # On demande au LLM (factice ou réel) de proposer des appels d'outils
            calls = call_llm(prompt, functions, USE_REAL_LLM)
            print("tool_calls:", calls)

            # On exécute chaque appel d'outil proposé, et on affiche le résultat
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])

In [ ]:
# On exécute l'Exercice 5 avec le prompt "Add 2 to 20"
# Résultat attendu : tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
#                    result: ['22']
await ex5_run("Add 2 to 20")

## Optionnel — outil multiply(a, b)

L'outil `multiply(a, b)` a déjà été ajouté dans `server.py` (voir plus haut). Pour le tester, il suffit de relancer `ex5_run` avec un prompt de multiplication :

In [ ]:
# On teste l'outil multiply avec un nouveau prompt
# Résultat attendu : tool_calls: [{'name': 'multiply', 'args': {'a': 3, 'b': 4}}]
#                    result: ['12']
await ex5_run("Multiply 3 times 4")